In [ ]:
import pandas as pd
import duckdb

In [ ]:
pd.set_option('display.max_rows', 20)      # shows ~first 10 + last 10
pd.set_option('display.min_rows', 30) 

In [ ]:
data_dir = "../csv"  
ai="grok"

sales = pd.read_csv(f"{data_dir}/sales-{ai}.csv", parse_dates=['DATE'])
sales

In [ ]:
parts = pd.read_csv(f"{data_dir}/parts.csv")
parts['SQFT'] = parts['SQFT'].astype('Int64')
parts

In [ ]:
duckdb.query("DROP VIEW IF EXISTS sales_monthly_view");

duckdb.query("""
    CREATE VIEW sales_monthly_view AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        licensee,
        part_id,
        CAST(SUM(qty) AS INT64) AS tot_qty, 
        CAST(SUM(amt) AS INT64) AS tot_amt,
    FROM sales
    GROUP BY month, supplier, licensee, part_id
""")

print("\nCreated `monthly_sales_view`.\n") 
print("Is makes getting supplier/license monthly or YTD totals easier.\n") 

In [ ]:
monthly_sales = duckdb.query(
    """
    SELECT * 
    FROM sales_monthly_view 
    ORDER BY month, supplier, licensee, part_id
    """).df()

print("\nHere's what the monthly view looks like.\n") 

monthly_sales

In [ ]:
df = duckdb.query("""
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        printf('%,d', SUM(amt)) as tot_amt

    FROM sales
    GROUP BY month, supplier
    ORDER BY month, supplier
""").df()

print("\n\nSupplier Monthly Totals - simple SQL query\n")

df

In [ ]:
df = duckdb.query("""
    SELECT 
        strftime('%Y-%m', date) AS month,
        licensee,
        printf('%,d', SUM(amt)) as tot_amt
    FROM sales
    GROUP BY month, licensee
    ORDER BY month, licensee
""").df()

print("\n\nLicensee Monthly Totals - simple SQL query\n")

df

In [ ]:
sql = """ --grok v2 licensee
SELECT 
    month,
    licensee,
--    SUM(tot_amt) OVER (PARTITION BY licensee ORDER BY month) AS ytd_amt
    printf('%,d', SUM(tot_amt) OVER (PARTITION BY licensee ORDER BY month)) AS ytd_amt
FROM (
    SELECT month, licensee, SUM(tot_amt) AS tot_amt
    FROM sales_monthly_view
    GROUP BY month, licensee
)
ORDER BY month, licensee;
"""


df = duckdb.query(sql).df()

print("\n\nLicensee monthly YTD\n")

df

In [ ]:
sql = """ --grok v2 supplier
SELECT 
    month,
    supplier,
    printf('%,d', tot_amt) as month_tot,
    printf('%,d', SUM(tot_amt) OVER (PARTITION BY supplier ORDER BY month)) AS ytd_amt
FROM (
    SELECT month, supplier, SUM(tot_amt) AS tot_amt
    FROM sales_monthly_view
    GROUP BY month, supplier
)
WHERE supplier = 'nippon-metal'
ORDER BY month, supplier;
"""

df = duckdb.query(sql).df()

print("\nSupplier monthly YTD\n")


# i like this, cuz (as requested) i only want to change it this one time.
with pd.option_context('display.max_rows', len(df)):
    display(df)

In [ ]:
df